In [ ]:
import os
import re
from dotenv import load_dotenv
from llama_index.core import (
    Settings, 
    VectorStoreIndex, 
    SimpleDirectoryReader, 
    StorageContext, 
    load_index_from_storage
)
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import MarkdownNodeParser

load_dotenv()

Settings.llm = OpenAI(temperature=1, model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))




In [ ]:
import re

def extract_metadata_from_filename(file_path):
   
    file_name = os.path.basename(file_path)
    
    metadata = {
        "file_name": file_name,
        "law_year": "unknown",
        "law_number": "unknown",
        "category": "قانون "
    }
    
   
    year_match = re.search(r'لسنة\s*(\d{4})', file_name)
    if year_match:
        metadata["law_year"] = int(year_match.group(1))
        
   
    number_match = re.search(r'رقم\s*(\d+)', file_name)
    if number_match:
        metadata["law_number"] = int(number_match.group(1))
        
    
    return metadata


test_file = "D:\Qanoon\data\القانون رقم 32 لسنة 2001 م بشأن اعتماد ميزانية التحول للسنة المالية 1370 و.ر-ساري (نافذ)-32-مؤتمر الشعب العام-المالية-الميزانية-28-12-2001 (1).md"
print(extract_metadata_from_filename(test_file))

{'file_name': 'القانون رقم 32 لسنة 2001 م بشأن اعتماد ميزانية التحول للسنة المالية 1370 و.ر-ساري (نافذ)-32-مؤتمر الشعب العام-المالية-الميزانية-28-12-2001 (1).md', 'law_year': 2001, 'law_number': 32, 'category': 'legal'}


In [3]:
from llama_index.core import SimpleDirectoryReader


INPUT_DIR = "./data" 

reader = SimpleDirectoryReader(
    input_dir=INPUT_DIR,
    required_exts=[".md"],
    recursive=True,
    file_metadata=extract_metadata_from_filename 
)

documents = reader.load_data()
print(f"{len(documents)}")

3


In [4]:
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter


node_parser = MarkdownNodeParser()


pipeline = IngestionPipeline(
    transformations=[
        node_parser,
        Settings.embed_model,
    ]

)


In [7]:
nodes = pipeline.run(documents=documents)
print(f"{len(nodes)}")
    

24
